# 08 — RQ5: On Track for 100% Renewable Electricity by 2030?

**Question:** Is Austria on track to reach 100% renewable *electricity* (national
balance, annual net) by 2030, the target of the Renewable Expansion Act
(EAG — Erneuerbaren-Ausbau-Gesetz, 2021)?

Strictly the renewable-**electricity** question. The greenhouse-gas trajectory is
RQ6, kept separate by design.

Primary source: **`owid_energy_at`** — `renewables_share_elec` (annual %), with
**ENTSO-E** generation as a cross-check. Method: a trend fit extrapolated to 2030
with an uncertainty band, compared against the 100% target.

Two forks settled *after* this EDA (Step 1), not assumed:

| Fork | Options | Decided |
|---|---|---|
| Trend window | how far back the fit starts (2019–24 too short to extrapolate) | Step 2, from the curve |
| Functional form | log-linear vs logit — a share is bounded at 100% | Step 2, from the shape |


## Setup

In [ ]:
# imports, paths, house style, read-only DuckDB connection
import sys
from pathlib import Path
from functools import partial

import duckdb
import matplotlib.pyplot as plt
from IPython.display import display
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.viz import set_house_style, PALETTE, line_profile, save_headline_fig, save_qa_fig

set_house_style()
save_qa = partial(save_qa_fig, notebook="08_rq5_renewable_electricity")

# read_only=True — RQ notebooks only SELECT; a stray write raises instead of mutating the DB
DB_PATH = PROJECT_ROOT / "data" / "processed" / "austria_energy.duckdb"
con = duckdb.connect(str(DB_PATH), read_only=True)

## Step 1 — profile the series

Find the usable span, read the shape.

### 1 · Coverage — usable span & recent tail

In [ ]:
# usable span of renewables_share_elec
# The table spans the full century, but the *share* column only has values once
# OWID has electricity data for AT. Same FILTER pattern as the solar query (nb 03,
# Section 6): one aggregate, conditioned rows. COUNT(col) auto-skips nulls.

display(con.sql("""
SELECT
    COUNT(*)                                                   AS rows_total,
    COUNT(renewables_share_elec)                              AS rows_with_share,
    MIN(year) FILTER (WHERE renewables_share_elec IS NOT NULL) AS first_share_year,
    MAX(year) FILTER (WHERE renewables_share_elec IS NOT NULL) AS last_share_year
FROM owid_energy_at
""").df())

In [ ]:
# the recent tail — where does the series really end?
# Confirms whether a 2025 row exists and looks complete (gotcha #7: 2025 is
# provisional — RQ1 excluded it; RQ5 may include it as a deliberate extension).
# A suspiciously low total_twh/solar/wind in 2025 = partial-year provisional figure.

display(con.sql("""
SELECT year,
       renewables_share_elec  AS ren_share,
       electricity_generation AS total_twh,
       solar_electricity      AS solar_twh,
       wind_electricity       AS wind_twh,
       hydro_electricity      AS hydro_twh
FROM owid_energy_at
WHERE year >= 2018
ORDER BY year
""").df())

### 2 · Full non-null share history (for plotting)

In [ ]:
df_ren = con.sql("""
SELECT year, renewables_share_elec AS ren_share
FROM owid_energy_at
WHERE renewables_share_elec IS NOT NULL
ORDER BY year
""").df()

print(f"{len(df_ren)} years: {df_ren['year'].min()}–{df_ren['year'].max()}")
df_ren.head()

### 3 · Eyeball the curve — decide the trend window

In [ ]:
# left: full history (long-run shape — hydro plateau vs recent rise).
# right: 2000 onward, y-zoomed (read the recent slope: linear / accelerating /
# back-loaded, like RQ1's ~5× solar gain). The final figure in Section 8 will
# anchor y at 0–100 for honesty; here we zoom only to judge the shape.

fig, (ax_full, ax_recent) = plt.subplots(1, 2, figsize=(13, 5))

# left: full history
line_profile(ax_full, df_ren["year"], df_ren["ren_share"], color=PALETTE["hydro"])
ax_full.axhline(100, color=PALETTE["accent"], ls="--", lw=1, label="100% target")
ax_full.axvspan(2019, 2024, color=PALETTE["muted"], alpha=0.18, label="RQ1 window")
ax_full.set(title="Full OWID history", xlabel="year",
            ylabel="renewable share of electricity (%)")
ax_full.set_ylim(0, 105)
ax_full.legend(loc="lower right")

# right: candidate window (2000+), y auto-scaled to read curvature
recent = df_ren[df_ren["year"] >= 2000]
line_profile(ax_recent, recent["year"], recent["ren_share"], color=PALETTE["hydro"])
ax_recent.axvspan(2019, 2024, color=PALETTE["muted"], alpha=0.18)
ax_recent.set(title="2000 onward (candidate window, y zoomed)", xlabel="year")

fig.suptitle("Renewable share of electricity — where should the trend fit begin?",
             fontsize=13)
plt.tight_layout()
save_qa(fig, "austria_renewable_share_of_electricity_trend_window")
plt.show()

## Step 2 — decisions, then the fit

From the curve (Step 1):

- **Fit window: 2010–2025** — the modern wind→solar expansion era. Excludes the
  pre-2005 hydro plateau (a no-expansion-policy regime that would flatten the slope).
- **2025 kept inside the fit** (revised — note below), flagged provisional in the figure.
- **Functional form: logit (log-odds), primary** — a share is bounded at 100%, and
  logit makes 100% an asymptote. Log-linear is shown as a foil (it can exceed 100%).
- **Sensitivity** across full (1985) / 2000+ / 2019+ windows answers "why this window?"

> **Why 2025 is *inside* the fit.** Hydro's water-year swing (RQ1: ±11 TWh) makes any
> single year noisy. 2024 was **high-water** (45.7 TWh hydro), flattering the share;
> 2025 was **low-water** (37.9 TWh), depressing it — solar actually *rose* between them.
> Ending the fit on 2024 alone would anchor the slope to a wet year and bias the
> extrapolation optimistic. Spanning wet-2024 and dry-2025 averages across the noise —
> more robust *and* more conservative. The cost (2025 is provisional, unverifiable vs
> ENTSO-E) is handled by flagging it; the verdict survives a sub-percentage-point revision.

**Scope caveat (carried into the finding).** OWID's `renewables_share_elec` is a
*generation* share. The Renewable Expansion Act (EAG) target is the *national net
balance* — annual renewable generation ≥ annual consumption, trade netted over the year,
not a generation share and not hourly. We track the generation share as the best
consistent annual proxy, and say so.

### 4 · Build the fit frame

In [ ]:
# build from df_ren (Section 2) — no re-query needed.
# Add modelling columns: the share as a fraction, and its logit (log-odds).
# logit(p) = ln(p / (1-p)) maps a (0,1) share onto the whole real line, so a
# straight-line trend there back-transforms to an S-curve that asymptotes at 100%.

fit_lo, fit_hi = 2010, 2025  # primary fit window (inclusive)

frame = df_ren.copy()
frame["p"] = frame["ren_share"] / 100.0  # percent → fraction
frame["logit_p"] = np.log(frame["p"] / (1.0 - frame["p"]))  # log-odds
frame["yr_c"] = frame["year"] - fit_lo  # centre at window start
frame["in_fit"] = frame["year"].between(fit_lo, fit_hi)  # fit mask
frame["is_2025"] = frame["year"] == 2025  # provisional flag

fit_df = frame[frame["in_fit"]].copy()
print(f"fit window {fit_lo}–{fit_hi}: {len(fit_df)} points")
display(fit_df[["year", "ren_share", "p", "logit_p", "yr_c"]].round(4))

### 5 · Fit the trend — logit vs log-linear

In [ ]:
# logit (primary) vs log-linear (foil).
# Plain OLS standard errors here: at n=16 you can't reliably estimate an
# autocorrelation structure, so HAC errors — which RQ4 needed across 52k hours —
# would be false precision. The real uncertainty is the window/form spread (later
# cells), not this parametric band.

# back-transforms: log-odds → share %, and unbounded log-share → share %
def inv_logit(x):
    return 100.0 / (1.0 + np.exp(-x))

def inv_log(x):
    return 100.0 * np.exp(x)

# primary — logit-linear: ln(p/(1-p)) ~ year. 100% is an asymptote it can't cross.
fit_logit = smf.ols("logit_p ~ yr_c", data=fit_df).fit()

# foil — log-linear: ln(p) ~ year. Constant growth rate → can punch past 100%.
fit_df = fit_df.assign(log_p=np.log(fit_df["p"]))
fit_loglin = smf.ols("log_p ~ yr_c", data=fit_df).fit()

# 2030 predictions
yr30 = 2030 - fit_lo
share30_logit = inv_logit(fit_logit.params["Intercept"] + fit_logit.params["yr_c"] * yr30)
share30_loglin = inv_log(fit_loglin.params["Intercept"] + fit_loglin.params["yr_c"] * yr30)

# logit slope → annual growth in the renewable-to-fossil odds
b = fit_logit.params["yr_c"]

# rate context: recent pace vs the pace needed to reach 100% by 2030
s2010 = frame.loc[frame.year == 2010, "ren_share"].iloc[0]
s2025 = frame.loc[frame.year == 2025, "ren_share"].iloc[0]
rate_recent = (s2025 - s2010) / (2025 - 2010)  # avg percentage-point/yr over the window
rate_needed = (100 - s2025) / (2030 - 2025)  # percentage-point/yr to close the gap

print("LOGIT (primary)")
print(f"  slope              {b:+.4f} log-odds/yr  →  odds ×{np.exp(b):.3f} per year")
print(f"  fitted 2010 share  {inv_logit(fit_logit.params['Intercept']):.1f}%")
print(f"  predicted 2030     {share30_logit:.1f}%   (gap to 100%: {100 - share30_logit:.1f} pp)")
print("\nLOG-LINEAR (foil)")
print(f"  predicted 2030     {share30_loglin:.1f}%   (unbounded form)")
print("\nRATE CHECK")
print(f"  recent pace        {rate_recent:.2f} pp/yr   (avg over 2010–2025)")
print(f"  needed for 100%    {rate_needed:.2f} pp/yr   →  {rate_needed / rate_recent:.1f}× the recent pace")

### 6 · Cross-check magnitudes — ENTSO-E vs OWID

In [ ]:
# renewable TWh and total TWh, ENTSO-E vs OWID.
#
# Before trusting OWID (the source RQ5's forecast rests on), rebuild the same
# magnitudes from our own ENTSO-E generation data and see if they agree.
# We look at the two pieces separately — renewable TWh, then total TWh — because
# a share (renewable ÷ total) can match by luck even when the pieces don't.
#
# ENTSO-E renewable = Solar + Wind Onshore + Biomass + all Hydro (includes pumped
# storage generation). Supply side only (flow='generation'), which also drops the
# pumped-storage pumping rows. SUM(mw) over hourly rows = MWh; ÷ 1e6 = TWh.

# ENTSO-E side: our own generation, summed to annual TWh
entsoe = con.sql("""
WITH gen_local AS (
    SELECT
        EXTRACT(year FROM ts_utc AT TIME ZONE 'Europe/Vienna') AS year,
        fuel_type,
        mw
    FROM generation
    WHERE flow = 'generation'
)
SELECT
    year,
    ROUND(SUM(mw) FILTER (
        WHERE fuel_type IN ('Solar', 'Wind Onshore', 'Biomass')
           OR fuel_type LIKE 'Hydro%'
    ) / 1e6, 2)              AS entsoe_ren_twh,
    ROUND(SUM(mw) / 1e6, 2)  AS entsoe_total_twh
FROM gen_local
WHERE year BETWEEN 2019 AND 2024
GROUP BY year
ORDER BY year
""").df()

# OWID side: total is published; renewable is reconstructed (share × total)
# OWID gives a share and a total, not a renewable-TWh column — so we derive it.
owid = con.sql("""
SELECT
    year,
    ROUND(electricity_generation, 2)                                 AS owid_total_twh,
    ROUND(renewables_share_elec / 100.0 * electricity_generation, 2) AS owid_ren_twh
FROM owid_energy_at
WHERE year BETWEEN 2019 AND 2024
ORDER BY year
""").df()

# line them up, with the ENTSO-E − OWID difference on each magnitude
cmp = entsoe.merge(owid, on="year")
cmp["ren_diff_twh"] = (cmp["entsoe_ren_twh"] - cmp["owid_ren_twh"]).round(2)
cmp["total_diff_twh"] = (cmp["entsoe_total_twh"] - cmp["owid_total_twh"]).round(2)

cmp = cmp[[
    "year",
    "entsoe_ren_twh", "owid_ren_twh", "ren_diff_twh",  # renewable magnitude
    "entsoe_total_twh", "owid_total_twh", "total_diff_twh",  # total magnitude
]]
display(cmp)

### 7 · Window-sensitivity across four start years

In [ ]:
# refit the same logit model on four start years.
# The parametric confidence interval (Section 5 / the figure) only captures scatter
# around one window's line. The choice of start year is a bigger lever, so we
# vary it and read the spread of 2030 predictions — the real uncertainty band.
#
# All windows end at 2025 (only the start moves). Reuses `frame`, `inv_logit`,
# `smf` from the earlier cells; yr_c stays centred at 2010 so every fit predicts
# 2030 at the same yr_c = 20.

windows = [
    ("full (1985+)", 1985, 2025),
    ("2000+", 2000, 2025),
    ("2010+ (primary)", 2010, 2025),
    ("2019+", 2019, 2025),
]

fits_by_window = {}  # stash fits for the headline figure (Section 8)
rows = []
for name, lo, hi in windows:
    d = frame[frame["year"].between(lo, hi)]  # inclusive
    f = smf.ols("logit_p ~ yr_c", data=d).fit()
    fits_by_window[name] = (f, lo, hi)

    b = f.params["yr_c"]  # slope, log-odds/yr
    p2030 = inv_logit(f.params["Intercept"] + b * (2030 - 2010))

    rows.append({
        "window": name,
        "n_points": len(d),
        "slope (log-odds/yr)": round(b, 4),
        "odds × per yr": round(np.exp(b), 3),
        "pred. 2030 (%)": round(p2030, 1),
        "gap to 100% (pp)": round(100 - p2030, 1),
    })

sens = pd.DataFrame(rows)
display(sens)

### 8 · Headline — trajectory vs the 2030 target

In [ ]:
# reuses frame, fits_by_window, inv_logit from earlier cells.
# Layers: OWID history (2025 provisional) · primary logit fit + 95% confidence band ·
# the other three windows as a faint fan · the 100% EAG target · the gap to it.

PRED_YEAR = 2030
primary, p_lo, p_hi = fits_by_window["2010+ (primary)"]

# a window's fitted share at any years (yr_c is centred at 2010 for every fit)
def share_at(f, yrs):
    return inv_logit(f.params["Intercept"] + f.params["yr_c"] * (yrs - 2010))

# primary fit: trend + 95% confidence band (trajectory uncertainty), 2010 → 2030
gx = np.arange(p_lo, PRED_YEAR + 1)
sf = primary.get_prediction(pd.DataFrame({"yr_c": gx - 2010})).summary_frame(alpha=0.05)
fit_y = inv_logit(sf["mean"].to_numpy())
ci_lo = inv_logit(sf["mean_ci_lower"].to_numpy())
ci_hi = inv_logit(sf["mean_ci_upper"].to_numpy())
p2030 = float(share_at(primary, np.array([PRED_YEAR]))[0])
gap = 100 - p2030

fig, ax = plt.subplots(figsize=(12, 6.5))

# other windows, faint fan + small 2030 endpoint dots (skip primary)
for i, (name, (f, lo, hi)) in enumerate(
        (nv for nv in fits_by_window.items() if "primary" not in nv[0])):
    xs = np.arange(lo, PRED_YEAR + 1)
    ax.plot(xs, share_at(f, xs), color=PALETTE["muted"], lw=1.0, ls=(0, (1, 1)),
            alpha=0.65, zorder=2,
            label="alt. windows (full / 2000+ / 2019+)" if i == 0 else None)
    ax.plot(PRED_YEAR, share_at(f, np.array([PRED_YEAR]))[0], "o",
            ms=3.5, color=PALETTE["muted"], alpha=0.8, zorder=4)

# 100% target
ax.axhline(100, color=PALETTE["accent"], ls="--", lw=1.5, zorder=2)
ax.annotate("100% — Renewable Expansion Act (EAG) 2030 target", (1985, 100),
            xytext=(2, 4), textcoords="offset points", fontsize=9,
            color=PALETTE["accent"])

# primary fit: confidence band + trend line
ax.fill_between(gx, ci_lo, ci_hi, color=PALETTE["price"], alpha=0.15, zorder=3,
                label="95% confidence band")
ax.plot(gx, fit_y, color="#1A1A1A", lw=2.2, zorder=5,
        label="logit trend (2010–2025, primary)")

# OWID history: solid to 2024, dashed bridge + hollow marker for 2025
hist = frame[frame["year"] <= 2024]
ax.plot(hist["year"], hist["ren_share"], color=PALETTE["hydro"], lw=1.6,
        marker="o", ms=4, zorder=4, label="OWID renewable share (history)")
y24, y25 = (float(frame.loc[frame.year == y, "ren_share"].iloc[0]) for y in (2024, 2025))
ax.plot([2024, 2025], [y24, y25], color=PALETTE["hydro"], lw=1.6, ls="--", zorder=4)
ax.plot(2025, y25, "o", ms=6, mfc="white", mec=PALETTE["hydro"], mew=1.6, zorder=6)
ax.annotate("2025\n(provisional)", (2025, y25), xytext=(0, -30),
            textcoords="offset points", ha="center", fontsize=8, color=PALETTE["demand"])

# the gap: primary 2030 point + double-arrow to 100%
ax.plot(PRED_YEAR, p2030, "o", ms=7, color="#1A1A1A", zorder=6)
ax.annotate(f"{p2030:.1f}%", (PRED_YEAR, p2030), xytext=(-6, -13),
            textcoords="offset points", ha="right", fontsize=9)
ax.annotate("", (PRED_YEAR, 100), (PRED_YEAR, p2030),
            arrowprops=dict(arrowstyle="<->", color=PALETTE["demand"], lw=1.3), zorder=7)
ax.annotate(f"{gap:.1f} pp short\nof target", (PRED_YEAR, (p2030 + 100) / 2),
            xytext=(7, 0), textcoords="offset points", va="center",
            fontsize=9, color=PALETTE["demand"], fontweight="normal")

ax.set(xlim=(1984, 2033), ylim=(58, 103),
       xlabel="year", ylabel="renewable share of electricity (%)")
ax.set_title("Austria's renewable-electricity trajectory lands ~12 points short of its 2030 target\n"
             "logit extrapolation of the OWID share, robust across four trend windows", fontsize=13)
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
save_headline_fig(fig, "rq5_renewable_electricity_2030")
plt.show()

In [ ]:
# Close the connection
con.close()

## Finding

**Finding (RQ5).** A bound-respecting **logit** extrapolation of Austria's renewable-electricity share
(primary window 2010–2025) projects **87.5% by 2030 — about 12.5 percentage points (pp)
short** of the Renewable Expansion Act's 100% target, and the shortfall is **robust to
the trend window**: every defensible fit lands in the low-to-high 80s (full history 80%
→ steepest window 90%), none reaching 100%. Closing the gap would demand roughly
**3.3 percentage points per year — nearly triple the ~1.15 pp/yr** achieved over
2010–2025, so on current momentum Austria is **off track** — with the honest caveat that
OWID's *generation* share is a proxy for the EAG's *national-net-balance* metric, and a
structural step-change in solar build-out could still bend the curve above trend.